# Rock energy labeling: log-pseudo-MSE only + energy-type Plotly surfaces

Вход: `united.csv`.

Notebook делает финальный упрощённый pipeline выделения энергоёмкости:

1. создаёт energy-response признаки;
2. считает `pseudo_mse = energy_input_proxy / speed`;
3. строит `hardness_score_smooth` только по `log(pseudo_mse_roll_median_60)`;
4. размечает энергоёмкость через сегментные квантили;
5. сохраняет датасет с `rock_energy_type_final`;
6. строит Plotly-поверхности `pressure_axis × pressure_rotation → speed` для всех энергоёмкостей.

В этой версии убраны `expected_speed_from_controls`, `formation_residual` и участие `drilling_efficiency` в расчёте hardness.
`drilling_efficiency` остаётся только как диагностический показатель в итоговых таблицах, потому что он является обратной величиной к `pseudo_mse`.

Очистка датасета не выполняется: предполагается, что `united.csv` уже очищен.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

RANDOM_STATE = 42
EPS = 1e-6

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1. Загрузка данных

In [2]:
DATA_PATH = "../datasets/united.csv"

_df = pd.read_csv(DATA_PATH)
_df = _df.drop(columns=["Unnamed: 0"], errors="ignore")

if "depth_m" not in _df.columns:
    if "depth" not in _df.columns:
        raise ValueError("Missing required depth/depth_m column in united.csv")
    _df["depth_m"] = _df["depth"]

_df["depth_m"] = pd.to_numeric(_df["depth_m"], errors="coerce")
df = _df

required_cols = [
    "processing_time",
    "well_id",
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "speed",
    "depth_m",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"?? ??????? ???????: {missing}")

df = df.dropna(subset=["depth_m"]).copy()
df["processing_time"] = pd.to_datetime(df["processing_time"])
df = df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
print("Has depth:", "depth" in df.columns)
print("Has depth_m:", "depth_m" in df.columns)
display(df[required_cols].head())
display(df[["pressure_axis", "pressure_rotation", "rotation", "speed", "depth_m"]].describe(percentiles=[.01, .05, .5, .95, .99]))


Loaded: ../datasets/united.csv
Shape: (415049, 8)
Has depth: True
Has depth_m: True


,processing_time,well_id,pressure_axis,pressure_rotation,rotation,speed,depth_m
0,2025-08-24 10:09:50.980,19601,804,4551,74.256,0.002755,0.0606
1,2025-08-24 10:10:00.260,19601,763,3782,73.812,0.003030,0.0909
2,2025-08-24 10:10:05.199,19601,879,4407,73.512,0.006060,0.1212
3,2025-08-24 10:10:14.610,19601,721,3705,73.962,0.002755,0.1515
4,2025-08-24 10:10:34.197,19601,859,3883,74.256,0.001515,0.1818


,pressure_axis,pressure_rotation,rotation,speed,depth_m
count,415049.000000,415049.000000,415049.000000,415049.000000,415049.000000
mean,17473.667273,14134.146325,103.945315,0.013116,10.733979
std,4682.982816,3243.523440,13.370361,0.006615,7.873565
min,317.000000,784.000000,50.010000,0.001002,-0.848400
1%,3713.000000,6271.000000,64.980000,0.002755,0.363600
5%,6645.000000,8246.000000,81.750000,0.005050,1.212000
50%,18861.000000,14637.000000,103.158000,0.012120,9.696000
95%,22343.000000,18758.600000,138.474000,0.024240,23.482500
99%,23626.000000,20881.000000,139.020000,0.030300,40.511100
max,24872.000000,26318.000000,139.578000,0.038957,82.052400


## 2. Базовые признаки

In [3]:
df["dt"] = (df.groupby("well_id")["processing_time"].diff().dt.total_seconds())

df["dt"] = df["dt"].fillna(df["dt"].median())

df["total_pressure"] = df["pressure_axis"] + df["pressure_rotation"]

df["pressure_balance"] = (df["pressure_axis"] / (df["pressure_axis"] + df["pressure_rotation"] + EPS))

df["axis_over_rot_pressure"] = (df["pressure_axis"] / (df["pressure_rotation"] + EPS))

df["rot_pressure_over_axis"] = (df["pressure_rotation"] / (df["pressure_axis"] + EPS))

df["rotation_efficiency"] = (df["rotation"] / (df["pressure_rotation"] + EPS))

df["axis_x_rotation"] = df["pressure_axis"] * df["rotation"]
df["rot_pressure_x_rotation"] = df["pressure_rotation"] * df["rotation"]

df["energy_input_proxy"] = (df["pressure_axis"] + df["pressure_rotation"] * df["rotation"])

df["pseudo_mse"] = (df["energy_input_proxy"] / (df["speed"] + EPS))

# Диагностический показатель: почти обратная величина к pseudo_mse.
# В hardness_score он больше не используется, но остаётся в summary для проверки монотонности классов.
df["drilling_efficiency"] = (df["speed"] / (df["energy_input_proxy"] + EPS))

df["log_energy_input_proxy"] = np.log1p(df["energy_input_proxy"])
df["log_pseudo_mse"] = np.log1p(df["pseudo_mse"])

display(df[["dt", "energy_input_proxy", "pseudo_mse", "drilling_efficiency", "rotation_efficiency", "pressure_balance",]].describe(percentiles=[.01, .05, .5, .95, .99]))

,dt,energy_input_proxy,pseudo_mse,drilling_efficiency,rotation_efficiency,pressure_balance
count,415049.000000,4.150490e+05,4.150490e+05,4.150490e+05,415049.000000,415049.000000
mean,7.270602,1.494500e+06,1.467727e+08,9.279041e-09,0.007803,0.545942
std,110.393457,4.009753e+05,9.155477e+07,5.931875e-09,0.002483,0.057506
min,0.089000,4.174649e+04,3.009300e+06,3.870720e-10,0.002104,0.013293
1%,0.229000,4.697380e+05,3.257078e+07,2.153750e-09,0.004789,0.320485
5%,0.464000,7.635049e+05,5.221720e+07,3.292781e-09,0.005407,0.428681
50%,5.135000,1.542361e+06,1.239899e+08,8.064516e-09,0.007111,0.558077
95%,10.891000,2.119089e+06,3.036335e+08,1.914947e-08,0.012153,0.607300
99%,23.689560,2.462635e+06,4.641533e+08,3.070111e-08,0.015835,0.629086
max,42583.603000,3.668551e+06,2.581497e+09,3.322895e-07,0.121481,0.933514


## 3. Rolling-признаки

In [4]:
def add_rolling_stats(data, cols, windows=(12, 30, 60), group_col="well_id"):
    out = data.copy()

    for col in cols:
        for w in windows:
            min_p = max(3, w // 3)

            out[f"{col}_roll_median_{w}"] = (
                out.groupby(group_col)[col]
                   .transform(lambda s: s.rolling(w, min_periods=min_p).median())
            )

            out[f"{col}_roll_mean_{w}"] = (
                out.groupby(group_col)[col]
                   .transform(lambda s: s.rolling(w, min_periods=min_p).mean())
            )

            out[f"{col}_roll_std_{w}"] = (
                out.groupby(group_col)[col]
                   .transform(lambda s: s.rolling(w, min_periods=min_p).std())
            )

    return out

rolling_cols = [
    "energy_input_proxy",
    "pseudo_mse",
    # drilling_efficiency не участвует в hardness_score, но rolling-версия остаётся
    # только для диагностической summary-таблицы.
    "drilling_efficiency",
    "rotation_efficiency",
    "speed",
    "rotation",
    "pressure_balance",
]

df = add_rolling_stats(df, rolling_cols, windows=(12, 30, 60))

print("Rolling features added:", len([c for c in df.columns if "_roll_" in c]))

Rolling features added: 63


## 4. Continuous hardness score: только `log_pseudo_mse`


In [5]:
def zscore(s):
    return (s - s.mean()) / (s.std() + EPS)

# Финальная формула энергоёмкости.
# Теперь score строится только на proxy энергоёмкости:
# сколько условной энергии требуется на единицу скорости проходки.
#
# Чем выше pseudo_mse, тем выше сопротивление бурению / энергоёмкость.
df["hardness_score"] = zscore(df["log_pseudo_mse"])

df["hardness_score_smooth"] = zscore(
    np.log1p(df["pseudo_mse_roll_median_60"])
)

print("Hardness formula:")
print("hardness_score_smooth = zscore(log1p(pseudo_mse_roll_median_60))")

display(df[[
    "hardness_score",
    "hardness_score_smooth",
    "log_pseudo_mse",
    "pseudo_mse_roll_median_60",
]].describe(percentiles=[.01, .05, .5, .95, .99]))


Hardness formula:
hardness_score_smooth = zscore(log1p(pseudo_mse_roll_median_60))


,hardness_score,hardness_score_smooth,log_pseudo_mse,pseudo_mse_roll_median_60
count,4.150490e+05,3.826350e+05,415049.000000,3.826350e+05
mean,-4.312468e-15,4.297480e-15,18.649861,1.300496e+08
std,9.999982e-01,9.999975e-01,0.552515,5.068067e+07
min,-6.755725e+00,-6.033536e+00,14.917218,1.117867e+07
1%,-2.445063e+00,-2.697314e+00,17.298926,4.165734e+07
5%,-1.590795e+00,-1.737829e+00,17.770923,6.081292e+07
50%,-2.561101e-02,8.954485e-02,18.635711,1.250026e+08
95%,1.595377e+00,1.660294e+00,19.531332,2.322187e+08
99%,2.363488e+00,2.115928e+00,19.955725,2.779208e+08
max,5.469120e+00,3.724039e+00,21.671635,5.239589e+08


## 5. Финальная разметка энергоёмкости: `energy_type_segment_quantile`

In [6]:
energy_labels_4 = [
    "soft_low_energy",
    "medium_low_energy",
    "medium_high_energy",
    "hard_high_energy",
]

SEGMENT_SIZE = 60

segment_df = df.dropna(subset=[
    "hardness_score_smooth",
    "pseudo_mse_roll_median_60",
    "drilling_efficiency_roll_median_60",
    "speed",
]).copy()

segment_df = segment_df.sort_values(["well_id", "processing_time"]).copy()
segment_df["_row_in_well"] = segment_df.groupby("well_id").cumcount()
segment_df["segment_id"] = (segment_df["_row_in_well"] // SEGMENT_SIZE).astype(int)

segments = (
    segment_df
    .groupby(["well_id", "segment_id"])
    .agg(
        segment_start=("processing_time", "min"),
        segment_end=("processing_time", "max"),
        rows=("speed", "size"),
        hardness_segment=("hardness_score_smooth", "median"),
        pseudo_mse_segment=("pseudo_mse_roll_median_60", "median"),
        efficiency_segment=("drilling_efficiency_roll_median_60", "median"),
        speed_segment=("speed", "median"),
    )
    .reset_index()
)

segments["energy_type_segment_quantile"] = pd.qcut(
    segments["hardness_segment"],
    q=4,
    labels=energy_labels_4,
    duplicates="drop",
).astype(str)

segment_df = segment_df.merge(
    segments[["well_id", "segment_id", "hardness_segment", "energy_type_segment_quantile"]],
    on=["well_id", "segment_id"],
    how="left",
)

display(
    segments
    .groupby("energy_type_segment_quantile")
    .agg(
        segments=("segment_id", "size"),
        rows=("rows", "sum"),
        hardness_segment=("hardness_segment", "median"),
        pseudo_mse_segment=("pseudo_mse_segment", "median"),
        efficiency_segment=("efficiency_segment", "median"),
        speed_segment=("speed_segment", "median"),
    )
    .sort_values("hardness_segment")
)

,segments,rows,hardness_segment,pseudo_mse_segment,efficiency_segment,speed_segment
energy_type_segment_quantile,,,,,,
soft_low_energy,1801,96310,-1.072210,7.906382e+07,1.264761e-08,0.018180
medium_low_energy,1800,95634,-0.224097,1.104614e+08,9.053351e-09,0.012120
medium_high_energy,1800,94953,0.312110,1.364682e+08,7.328527e-09,0.012120
hard_high_energy,1801,95738,1.105058,1.865598e+08,5.363636e-09,0.007162


## 6. Возвращаем финальную разметку в основной датасет

In [7]:
merge_cols = [
    "processing_time",
    "well_id",
    "segment_id",
    "hardness_segment",
    "energy_type_segment_quantile",
]

df_out = df.merge(
    segment_df[merge_cols],
    on=["processing_time", "well_id"],
    how="left",
    suffixes=("", "_segment"),
)

if "depth_m" not in df_out.columns:
    if "depth" not in df_out.columns:
        raise ValueError("df_out lost depth/depth_m before save")
    df_out["depth_m"] = df_out["depth"]
df_out["depth_m"] = pd.to_numeric(df_out["depth_m"], errors="coerce")

df_out["rock_energy_type_final"] = df_out["energy_type_segment_quantile"]

display(df_out[[
    "processing_time",
    "well_id",
    "depth_m",
    "speed",
    "energy_input_proxy",
    "pseudo_mse",
    "drilling_efficiency",
    "hardness_score_smooth",
    "segment_id",
    "hardness_segment",
    "rock_energy_type_final",
]].head())


,processing_time,well_id,depth_m,speed,energy_input_proxy,pseudo_mse,drilling_efficiency,hardness_score_smooth,segment_id,hardness_segment,rock_energy_type_final
0,2025-08-24 10:09:50.980,19601,0.0606,0.002755,338743.056,1.229314e+08,8.131666e-09,NaN,NaN,NaN,NaN
1,2025-08-24 10:10:00.260,19601,0.0909,0.003030,279919.984,9.235235e+07,1.082452e-08,NaN,NaN,NaN,NaN
2,2025-08-24 10:10:05.199,19601,0.1212,0.006060,324846.384,5.359617e+07,1.865497e-08,NaN,NaN,NaN,NaN
3,2025-08-24 10:10:14.610,19601,0.1515,0.002755,274750.210,9.970810e+07,1.002564e-08,NaN,NaN,NaN,NaN
4,2025-08-24 10:10:34.197,19601,0.1818,0.001515,289195.048,1.907619e+08,5.238679e-09,NaN,NaN,NaN,NaN


## 7. Сводка финальной разметки

In [8]:
summary_final = (
    df_out
    .groupby("rock_energy_type_final")
    .agg(
        rows=("speed", "size"),
        speed_median=("speed", "median"),
        pseudo_mse_smooth=("pseudo_mse_roll_median_60", "median"),
        efficiency_smooth=("drilling_efficiency_roll_median_60", "median"),
        hardness_smooth=("hardness_score_smooth", "median"),
        pressure_axis_median=("pressure_axis", "median"),
        pressure_rotation_median=("pressure_rotation", "median"),
        rotation_median=("rotation", "median"),
    )
    .sort_values("hardness_smooth")
)

display(summary_final)
print("Note: efficiency_smooth is diagnostic only. It is not used in hardness_score.")


,rows,speed_median,pseudo_mse_smooth,efficiency_smooth,hardness_smooth,pressure_axis_median,pressure_rotation_median,rotation_median
rock_energy_type_final,,,,,,,,
soft_low_energy,96310,0.01818,7.879613e+07,1.269129e-08,-1.080811,16809.0,13709.0,103.410
medium_low_energy,95634,0.01212,1.104952e+08,9.050232e-09,-0.223320,19420.0,15276.0,103.008
medium_high_energy,94953,0.01212,1.364211e+08,7.330147e-09,0.311234,20389.0,15534.0,102.966
hard_high_energy,95738,0.00606,1.872205e+08,5.345583e-09,1.114027,19764.0,14540.0,103.458


Note: efficiency_smooth is diagnostic only. It is not used in hardness_score.


## 8. Plotly surfaces для всех энергоёмкостей

In [9]:
surface_features = [
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "hardness_score_smooth",
    "pressure_balance",
    "axis_over_rot_pressure",
    "rot_pressure_over_axis",
    "rotation_efficiency",
    "energy_input_proxy",
]

SURFACE_HTML_DIR = Path("plotly_surfaces_html")
SURFACE_HTML_DIR.mkdir(exist_ok=True)

for surface_type in energy_labels_4:
    surface_train = df_out[df_out["rock_energy_type_final"] == surface_type].dropna(subset=[
        "pressure_axis", "pressure_rotation", "rotation", "hardness_score_smooth", "speed"
    ]).copy()

    if len(surface_train) < 500:
        print("Skip small class:", surface_type, len(surface_train))
        continue

    surface_model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=RANDOM_STATE,
        ))
    ])

    surface_model.fit(surface_train[surface_features], surface_train["speed"])

    fixed_state = surface_train[surface_features].median()

    p_ax_grid = np.linspace(
        surface_train["pressure_axis"].quantile(0.05),
        surface_train["pressure_axis"].quantile(0.95),
        60,
    )

    p_rot_grid = np.linspace(
        surface_train["pressure_rotation"].quantile(0.05),
        surface_train["pressure_rotation"].quantile(0.95),
        60,
    )

    PA, PR = np.meshgrid(p_ax_grid, p_rot_grid)

    grid = pd.DataFrame({
        "pressure_axis": PA.ravel(),
        "pressure_rotation": PR.ravel(),
    })

    for col in surface_features:
        if col not in grid.columns:
            grid[col] = fixed_state[col]

    grid["pressure_balance"] = grid["pressure_axis"] / (grid["pressure_axis"] + grid["pressure_rotation"] + EPS)
    grid["axis_over_rot_pressure"] = grid["pressure_axis"] / (grid["pressure_rotation"] + EPS)
    grid["rot_pressure_over_axis"] = grid["pressure_rotation"] / (grid["pressure_axis"] + EPS)
    grid["rotation_efficiency"] = grid["rotation"] / (grid["pressure_rotation"] + EPS)
    grid["energy_input_proxy"] = grid["pressure_axis"] + grid["pressure_rotation"] * grid["rotation"]

    Z = surface_model.predict(grid[surface_features]).reshape(PA.shape)

    fig = go.Figure()
    fig.add_trace(go.Surface(
        x=PA,
        y=PR,
        z=Z,
        colorscale="Viridis",
        opacity=0.95,
        showscale=True,
    ))

    fig.update_layout(
        title=f"Surface by energy type: {surface_type}",
        scene=dict(
            xaxis_title="pressure_axis",
            yaxis_title="pressure_rotation",
            zaxis_title="speed",
        ),
        height=800,
    )

    surface_html = SURFACE_HTML_DIR / f"surface_{surface_type}.html"
    fig.write_html(surface_html, include_plotlyjs=True, full_html=True)
    print("Saved browser surface:", surface_html)



Saved browser surface: plotly_surfaces_html\surface_soft_low_energy.html
Saved browser surface: plotly_surfaces_html\surface_medium_low_energy.html
Saved browser surface: plotly_surfaces_html\surface_medium_high_energy.html
Saved browser surface: plotly_surfaces_html\surface_hard_high_energy.html


## 9. Сохранение результата

In [10]:
OUTPUT_PATH = "united_rock_energy_segment_quantile.csv"
CONFIG_PATH = "rock_energy_segment_quantile_config.json"

if "depth_m" not in df_out.columns:
    if "depth" not in df_out.columns:
        raise ValueError("Missing depth/depth_m in final df_out")
    df_out["depth_m"] = df_out["depth"]
df_out["depth_m"] = pd.to_numeric(df_out["depth_m"], errors="coerce")

front_cols = [
    "processing_time",
    "depth_m",
    "rotation",
    "pressure_axis",
    "pressure_rotation",
    "well_id",
    "speed",
]
remaining_cols = [col for col in df_out.columns if col not in front_cols and col != "depth"]
df_out = df_out[front_cols + remaining_cols]

df_out.to_csv(OUTPUT_PATH, index=False)

config = {
    "data_path": DATA_PATH,
    "output_path": OUTPUT_PATH,
    "final_method": "energy_type_segment_quantile_log_pseudo_mse_only",
    "segment_size": SEGMENT_SIZE,
    "labels": energy_labels_4,
    "surface_features": surface_features,
    "depth_column": "depth_m",
    "hardness_score_formula": "zscore(log1p(pseudo_mse_roll_median_60))",
    "removed_from_hardness": [
        "expected_speed_from_controls",
        "formation_residual",
        "relative_formation_residual",
        "drilling_efficiency",
    ],
    "diagnostic_columns": {
        "drilling_efficiency": "kept only for validation/summary because it is inverse to pseudo_mse",
    },
    "interpretation": {
        "hardness_score_smooth": "continuous operational drilling resistance index based only on proxy energy per penetration",
        "rock_energy_type_final": "final discrete energy-response regime from segment-level quantile segmentation",
        "pseudo_mse": "proxy energy per penetration, not physical MSE",
    }
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_PATH)
print("Saved:", CONFIG_PATH)
print("Output has depth_m:", "depth_m" in df_out.columns)
print("Output columns head:", list(df_out.columns[:10]))
print("No expected-speed/residual model artifact is saved in this version.")


Saved: united_rock_energy_segment_quantile.csv
Saved: rock_energy_segment_quantile_config.json
Output has depth_m: True
Output columns head: ['processing_time', 'depth_m', 'rotation', 'pressure_axis', 'pressure_rotation', 'well_id', 'speed', 'dt', 'total_pressure', 'pressure_balance']
No expected-speed/residual model artifact is saved in this version.
